# Global Closed Economy - Prospective

## 0. Initialize context

In [ ]:
%run ../0-context/context.py

In [ ]:
dir_input = "../../data/1-EEIO_foundations/1-global_closed_economy/1-notebook_input/"
dir_output = "../../data/1-EEIO_foundations/1-global_closed_economy/2-notebook_output/"

### 0.1. Load accounts

In [ ]:
from matmat.core.accounts import builder as accounts_builder
path_to_accounts = os.path.join(dir_input, "accounts")
accounts_director = accounts_builder.get_director(reset=True)
accounts = accounts_director.make_from_path(path=path_to_accounts)
accounts.calculate()

## 1. Shock on the final demand

### 1.1. Initialize system shock consistent with accounts system

In [ ]:
from matmat.core.shocks.system import builder as system_shock_builder
system_shock_director = system_shock_builder.get_director(reset=True)
system_shock_director.set_regions(accounts.system.regions)
system_shock_director.set_sectors(accounts.system.sectors)
system_shock_director.set_final_demand_categories(accounts.system.final_demand_categories)

In [ ]:
# Initialize a system shock for final demand variation
system_shock = system_shock_director.make_shock_y()

In [ ]:
system_shock.id

In [ ]:
# Update identity fields
system_shock.id.base_year = accounts.system.id.base_year
system_shock.id.proj_year = 2050
system_shock.id.scenario_name="formation"

**Exercise:** Display final demand shock matrices

In [ ]:
# Display dY
system_shock.dY.df

### 1.2. Define the relative variations

**Exercise:** Set a 20% decrease to final demand 'Households' for sector 'Food processing'

In [ ]:
system_shock.dY.set_values(0.0)
system_shock.dY.set_value_at(
    value=-0.2,
    rows_filters={"sector": ["Food processing"]},
    columns_filters={"Y_category": ["Households"]}, 
)
system_shock.dY.df

### 1.3. Shock the system

**Exercise:** Apply the shock to the system

In [ ]:
# Duplicate references
accounts_calibrated = accounts.copy()
accounts_projected = accounts.copy()
# Apply the system shock to the accounts system
# First, reset the accounts (both system and extensions) with the method reset_for_shock()
accounts_projected.reset_for_shock()
# Then use the method shock() applicable to a system
accounts_projected.system.shock(system_shock)

**Exercise:** Display the changes on final demand data Y

In [ ]:
# Display Y from calibrated system
accounts.system.Y.df.xs(axis=0, level="sector", key="Food processing", drop_level=False)

In [ ]:
# Display Y from projected system
accounts_projected.system.Y.df.xs(axis=0, level="sector", key="Food processing", drop_level=False)

### 1.3. Recalculate accounts

We now want to propagate this modification to the whole accounts, by recalculating:
- the system
- the extension ghg_emissions

**Exercise:** Calculate the accounts

In [ ]:
# Calculate the projected accounts with the method calculate()
accounts_projected.calculate()

**Exercise:** Show the modification of ghg_emissions d_cba in projected accounts

In [ ]:
# Display ghg_emissions d_cba from calibrated accounts
accounts_calibrated.ghg_emissions.d_cba.df.xs(axis=1, level="sector", key="Food processing")

In [ ]:
# Display ghg_emissions d_cba from projected accounts
accounts_projected.ghg_emissions.d_cba.df.xs(axis=1, level="sector", key="Food processing")

### 1.4. Save accounts and shock

In [ ]:
# Use the method save_data() to save the accounts to local directory
accounts_projected.save_to_path(os.path.join(dir_output, "accounts", "accounts_shocked_by_dY"))

In [ ]:
# Use the method save_data() to save the shock to local directory
system_shock.save_to_path(os.path.join(dir_output, "shocks", "shock_dY", "system"))

## 2. Shock on the technical coefficients

### 2.1. Initialize system shock consistent with accounts system

In [ ]:
# Initialize a system shock for technical coefficients variation
system_shock = system_shock_director.make_shock_a()

**Exercise:** Display final demand shock matrices

In [ ]:
# Display dA
system_shock.dA.df

In [ ]:
# Update identity fields
system_shock.id.base_year = accounts.system.id.base_year
system_shock.id.proj_year = 2050
system_shock.id.scenario_name="formation"

### 2.2. Define the relative variations

**Exercise:** Define the variations on technical coefficients
- Set a 20% decrease to sector 'Food processing' on sector 'Livestock'
- Set a 20% increase to sector 'Food processing' on sector 'Crops'

In [ ]:
system_shock.dA.set_values(0.0)
system_shock.dA.set_value_at(
    value=-0.2,
    rows_filters={"sector": ["Livestock"]},
    columns_filters={"sector": ["Food processing"]}, 
)
system_shock.dA.set_value_at(
    value=0.2,
    rows_filters={"sector": ["Crops"]},
    columns_filters={"sector": ["Food processing"]}, 
)
system_shock.dA.df

### 2.3. Shock the system

**Exercise:** Apply the shock to the system

In [ ]:
# Duplicate references
accounts_calibrated = accounts.copy()
accounts_projected = accounts.copy()
# Apply the system shock to the accounts system
# First reset the accounts (both system and extensions) for shock with the method reset_for_shock()
accounts_projected.reset_for_shock()
# Then use the method shock() applicable to a system
accounts_projected.system.shock(system_shock)

**Exercise:** Display the changes on technical coefficients data A

In [ ]:
# Display A from calibrated system
accounts.system.A.df.xs(axis=1, level="sector", key="Food processing", drop_level=False).xs(axis=0, level="category", key="Agriculture", drop_level=False)

In [ ]:
# Display A from projected system
accounts_projected.system.A.df.xs(axis=1, level="sector", key="Food processing").xs(axis=0, level="category", key="Agriculture", drop_level=False)

### 2.3. Recalculate accounts

We now want to propagate this modification to the whole accounts, by recalculating:
- the system
- the extension ghg_emissions

**Exercise:** Calculate the accounts

In [ ]:
# Calculate the projected accounts
accounts_projected.calculate()

**Exercise:** Show the modification of ghg_emissions d_cba in projected accounts

In [ ]:
# Display ghg_emissions d_cba from calibrated accounts
accounts_calibrated.ghg_emissions.d_cba.df.xs(axis=1, level="sector", key="Livestock", drop_level=False).sum(axis=0)

In [ ]:
# Display ghg_emissions d_cba from projected accounts
accounts_projected.ghg_emissions.d_cba.df.xs(axis=1, level="sector", key="Livestock", drop_level=False).sum(axis=0)

### 2.4. Save accounts and shock

In [ ]:
# Use the method save_data() to save the accounts to local directory
accounts_projected.save_to_path(os.path.join(dir_output, "accounts", "accounts_shocked_by_dA"))

In [ ]:
# Use the method save_data() to save the shock to local directory
system_shock.save_to_path(os.path.join(dir_output, "shocks", "shock_dA", "system"))

## 3. Shock on the intensities

### 3.1. Initialize extension shock consistent with extension ghg_emissions

In [ ]:
# Initialize an extension shock for intensities variation
from matmat.core.shocks.extension import builder as extension_shock_builder
extension_shock_director = extension_shock_builder.get_director(reset=True)
extension_shock_director.set_regions(accounts.ghg_emissions.regions)
extension_shock_director.set_sectors(accounts.ghg_emissions.sectors)
extension_shock_director.set_final_demand_categories(accounts.ghg_emissions.final_demand_categories)
extension_shock_director.set_extension_categories(accounts.ghg_emissions.extension_categories)

ghg_emissions_shock = extension_shock_director.make_shock_s_x(name="ghg_emissions")

**Exercise:** Display intensities shock matrices

In [ ]:
# Display dS_x_dom
ghg_emissions_shock.dS_x_dom.df

In [ ]:
# Update identity fields
ghg_emissions_shock.id.base_year = accounts.ghg_emissions.id.base_year
ghg_emissions_shock.id.proj_year = 2050
ghg_emissions_shock.id.scenario_name="formation"

### 3.2. Define the relative variations

**Exercise:** Define the variations on technical coefficients
- Set a 20% decrease for (source 'ghg_other' / gas 'CH4') from sector 'Livestock'

In [ ]:
ghg_emissions_shock.dS_x_dom.set_values(0.0)
ghg_emissions_shock.dS_x_dom.set_value_at(
    value=-0.2,
    rows_filters={"source": ["ghg_other"], "gas": ["CH4"]},
    columns_filters={"sector": ["Livestock"]},
)
ghg_emissions_shock.dS_x_dom.df

### 3.3. Shock the extension ghg_emissions

**Exercise:** Apply the shock to the extension

In [ ]:
# Duplicate references
accounts_calibrated = accounts.copy()
accounts_projected = accounts.copy()
# Apply the extension shock to the ghg_emissions
# First reset the extension before shock
accounts_projected.ghg_emissions.reset_for_shock()
# Then use the method shock() applicable to an extension
accounts_projected.ghg_emissions.shock(ghg_emissions_shock)

**Exercise:** Display the changes on intensities S_x_dom

In [ ]:
# Display S_x_dom from calibrated ghg_emissions
accounts_calibrated.ghg_emissions.S_x_dom.df.xs(axis=1, level="sector", key="Livestock", drop_level=False).xs(axis=0, level="gas", key="CH4", drop_level=False)

In [ ]:
# Display S_x_dom from projected ghg_emissions
accounts_projected.ghg_emissions.S_x_dom.df.xs(axis=1, level="sector", key="Livestock", drop_level=False).xs(axis=0, level="gas", key="CH4", drop_level=False)

### 3.3. Recalculate ghg_emissions

**Exercise:** Calculate the projected ghg_emissions

In [ ]:
# Calculate the projected ghg_emissions
accounts_projected.calculate_extensions()

**Exercise:** Show the modification of ghg_emissions d_cba in projected accounts

In [ ]:
# Display ghg_emissions d_cba from calibrated accounts
accounts_calibrated.ghg_emissions.d_cba.df.xs(axis=1, level="sector", key="Livestock").xs(axis=0, level="gas", key="CH4", drop_level=False)

In [ ]:
# Display ghg_emissions d_cba from projected accounts
accounts_projected.ghg_emissions.d_cba.df.xs(axis=1, level="sector", key="Livestock").xs(axis=0, level="gas", key="CH4", drop_level=False)

### 3.4. Save ghg_emissions shock

In [ ]:
# Use the method save_data() to save the accounts to local directory
ghg_emissions_shock.save_to_path(os.path.join(dir_output, "shocks", "shock_on_dS_x_dom", "extensions"))

In [ ]:
accounts_projected.save_to_path(os.path.join(dir_output, "accounts", "accounts_shocked_by_dS_x_dom"))